# Attention and transformers
**Machine Learning · Week 9 (Deep learning: images, sequences and attention) · Day 5**

Lesson page: https://sara-aminpour.com/tutorials/ml/w9d5.html

**Attention** lets every position of a sequence look straight at every other position. Each position asks a
question (its **query**), every position offers a label (its **key**) and some content (its **value**).
The answer is a weighted average of the values, weighted by how well each key matches the query:
$$\text{attention}(Q, K, V) = \text{softmax}\!\left(\frac{QK^\top}{\sqrt{d}}\right) V$$
Here it is by hand, for four positions with 2-number queries, keys and values.

In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.set_num_threads(1)
words = ["the", "cat", "sat", "down"]
# made-up numbers (a network learns them): the first coordinate means "a thing", the second "an action"
Q = np.array([[0.0, 0.0], [0.3, 0.3], [1.5, 0.0], [0.0, 1.5]])   # what each word is looking for
K = np.array([[0.0, 0.0], [3.0, 0.5], [0.5, 3.0], [0.0, 1.0]])   # what each word offers to be found by
V = np.array([[0.0, 0.0], [1.0, 0.0], [0.0, 1.0], [0.2, 0.2]])   # what each word passes on

scores = Q @ K.T / np.sqrt(Q.shape[1])            # how well every query matches every key
weights = np.exp(scores) / np.exp(scores).sum(axis=1, keepdims=True)   # softmax: each row adds up to 1
out = weights @ V
for i, w in enumerate(words):
    print(f"{w:5s} attends " + "  ".join(f"{words[j]} {weights[i, j]:.2f}" for j in range(4)) + f"  -> {np.round(out[i], 2)}")
print('"sat" looks for a thing and finds "cat"; "down" looks for an action and finds "sat".')

## The same in PyTorch, for real sizes
In a network, $Q$, $K$ and $V$ come from the input through three learned linear layers. PyTorch has the formula
built in, and it agrees with ours.

In [ ]:
def attention(Q, K, V):
    d = Q.shape[-1]
    weights = torch.softmax(Q @ K.transpose(-2, -1) / d ** 0.5, dim=-1)
    return weights @ V, weights

x = torch.randn(2, 5, 16)                                  # 2 sequences, 5 positions, 16 numbers each
to_q, to_k, to_v = nn.Linear(16, 16), nn.Linear(16, 16), nn.Linear(16, 16)
ours, w = attention(to_q(x), to_k(x), to_v(x))
theirs = F.scaled_dot_product_attention(to_q(x), to_k(x), to_v(x))
print("output", tuple(ours.shape), "weights", tuple(w.shape), "same as PyTorch:", torch.allclose(ours, theirs, atol=1e-6))

## A tiny transformer that sorts
A transformer block is attention plus a small feed-forward network, each with a shortcut (the input is added
back) and a normalization. Attention itself doesn't know the order of positions, so we add a learned
**position embedding** to every input. Task: read 8 digits, output them sorted. One block, one attention head.

In [ ]:
L, VOCAB, D = 8, 10, 64

class Block(nn.Module):
    def __init__(self):
        super().__init__()
        self.attn = nn.MultiheadAttention(D, num_heads=1, batch_first=True)
        self.norm1, self.norm2 = nn.LayerNorm(D), nn.LayerNorm(D)
        self.ff = nn.Sequential(nn.Linear(D, 128), nn.ReLU(), nn.Linear(128, D))

    def forward(self, x):
        a, weights = self.attn(x, x, x)          # self-attention: queries, keys and values all come from x
        x = self.norm1(x + a)
        return self.norm2(x + self.ff(x)), weights

class Sorter(nn.Module):
    def __init__(self, positions=True):
        super().__init__()
        torch.manual_seed(0)
        self.tok = nn.Embedding(VOCAB, D)                    # a learned vector for each digit
        self.pos = nn.Parameter(torch.randn(L, D) * 0.1) if positions else None   # and for each position
        self.block = Block()
        self.out = nn.Linear(D, VOCAB)

    def forward(self, x):
        h = self.tok(x) + (self.pos if self.pos is not None else 0)
        h, weights = self.block(h)
        return self.out(h), weights

def train(model, steps=3000):
    opt = torch.optim.Adam(model.parameters(), lr=1e-3)
    gen = torch.Generator().manual_seed(0)
    for _ in range(steps):
        x = torch.randint(0, VOCAB, (64, L), generator=gen)
        opt.zero_grad()
        F.cross_entropy(model(x)[0].reshape(-1, VOCAB), x.sort(dim=1).values.reshape(-1)).backward()
        opt.step()
    return model

def test(model):
    x = torch.randint(0, VOCAB, (1000, L), generator=torch.Generator().manual_seed(1))
    with torch.no_grad():
        pred = model(x)[0].argmax(dim=-1)
    right = pred == x.sort(dim=1).values
    return right.float().mean().item(), right.all(dim=1).float().mean().item()

t0 = time.perf_counter()
sorter = train(Sorter())
digits_ok, lists_ok = test(sorter)
print(f"trained in {time.perf_counter() - t0:.0f} s: {digits_ok:.1%} of digits right, {lists_ok:.1%} of lists perfectly sorted")
example = torch.tensor([[7, 2, 9, 2, 0, 5, 3, 8]])
print("input: ", example[0].tolist())
print("output:", sorter(example)[0].argmax(dim=-1)[0].tolist())

## Where does it look?
Row $k$ of the attention matrix shows where output position $k$ looks while it decides which digit goes there.

In [ ]:
with torch.no_grad():
    _, weights = sorter(example)
fig, ax = plt.subplots(figsize=(5.4, 4.6))
ax.imshow(weights[0], cmap="Purples", vmin=0, vmax=weights.max())
ax.set_xticks(range(L), [str(d) for d in example[0].tolist()])
ax.set_yticks(range(L), [f"output {k + 1}" for k in range(L)])
ax.set_xlabel("input digits")
ax.set_title("Attention weights of the sorter")
ax.grid(False)
for i in range(L):
    for j in range(L):
        ax.text(j, i, f"{weights[0, i, j]:.2f}", ha="center", va="center", fontsize=7.5,
                color="white" if weights[0, i, j] > 0.35 else "#2c2842")
plt.show()

## Your turn: attention without positions
Train a `Sorter(positions=False)` and test it. Without position embeddings, every output position sees the
same set of digits in exactly the same way. Predict what happens before you run it!

In [ ]:
no_pos = None     # TODO: train(Sorter(positions=False))
# TODO: print test(no_pos), and the output for `example`

---
## Solution
Try the exercise yourself first: it's fine to get stuck for a minute. Then compare with this one (there are many good answers).

In [ ]:
no_pos = train(Sorter(positions=False))
print("digits right, lists right:", [round(v, 3) for v in test(no_pos)])
print("output for", example[0].tolist(), "->", no_pos(example)[0].argmax(dim=-1)[0].tolist())
print("Without positions, self-attention treats the input as a bag of digits: each output only knows its own")
print("digit and the whole bag, not which place in the sorted list it should fill. Order must be added.")